# LeetCode 2184 — Number of Ways to Build Sturdy Brick Wall bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-to-build-sturdy-brick-wall/ · Độ khó: **Medium** (🔒 Premium) · Chủ đề: Array, Dynamic Programming, Backtracking, Bit Manipulation, Bitmask

> **Lưu ý Premium:** đây là bài Premium; đề bài trong notebook được viết theo bản mô tả công khai của doocs/leetcode. Nếu bạn có tài khoản Premium, hãy **đối chiếu lại với đề gốc** (tên hàm, ràng buộc, ví dụ).

Skeleton LeetCode:

```python
class Solution:
    def buildWall(self, height: int, width: int, bricks: List[int]) -> int:
        ...
```

Mỗi hàng là một cách xếp gạch phủ đúng `width`; hai hàng kề nhau **không** được có mạch nối (khe giữa hai viên) ở cùng vị trí. Mã hóa mỗi hàng bằng **bitmask các vị trí mạch nối** ⇒ hai hàng hợp nhau ⇔ `a & b == 0`. Notebook đi từ **vét cạn mọi bức tường** → **DP theo hàng với mọi cặp hàng** → **DP với danh sách kề tính sẵn** → **DP + biến đổi tổng tập con (SOS / zeta transform)** → **lũy thừa ma trận** (cho `height` lớn).

## 1. Mục tiêu học

1. Liệt kê mọi **cách xếp một hàng** (hợp thành có thứ tự — composition) bằng quay lui, và mã hóa bằng **bitmask mạch nối** `width − 1` bit.
2. Biến điều kiện "không trùng mạch nối" thành phép `AND` bằng 0.
3. Viết DP theo chiều cao: `dp[h][m]` = số tường cao `h` có hàng trên cùng là `m`.
4. Tăng tốc bước chuyển bằng **tổng trên tập con** (SOS DP): `Σ_{p ⊆ ~m} dp[p]` cho mọi `m` trong `O(2^k · k)`.
5. Dùng **lũy thừa ma trận** khi chiều cao rất lớn, và hiểu cái giá `O(R³ log h)`.

## 2. Đề bài & ví dụ

Xây tường cao `height`, rộng `width`. Có các loại gạch cao 1, rộng `bricks[i]` (các giá trị **khác nhau**), số lượng vô hạn, không được xoay. Mỗi hàng phải dài **đúng** `width`. Tường **vững** nếu hai hàng kề nhau **không** có mạch nối ở cùng vị trí (trừ hai đầu tường). Đếm số cách xây tường vững, modulo `10⁹ + 7`.

**Ví dụ 1:** `height = 2, width = 3, bricks = [1,2]` → `2` (hàng `[1,2]` trên `[2,1]` và ngược lại; `[1,1,1]` có mạch ở 1 và 2 nên không hợp với hàng nào).

**Ví dụ 2:** `height = 1, width = 1, bricks = [5]` → `0`.

**Ràng buộc:** `1 ≤ height ≤ 100`, `1 ≤ width ≤ 10`, `1 ≤ len(bricks) ≤ 10`, `1 ≤ bricks[i] ≤ 10`, các `bricks[i]` khác nhau.

In [1]:
import random
import time
from itertools import product
from functools import lru_cache
from typing import List

MOD = 10**9 + 7
examples = [((2, 3, [1, 2]), 2), ((1, 1, [5]), 0)]

def draw_row(row, width):
    cells = []
    for b in row:
        cells.append("[" + "=" * (2 * b - 1) + "]")
    return "".join(cells)

print("hàng [1,2]:", draw_row([1, 2], 3), "| hàng [2,1]:", draw_row([2, 1], 3), "| hàng [1,1,1]:", draw_row([1, 1, 1], 3))

hàng [1,2]: [=][===] | hàng [2,1]: [===][=] | hàng [1,1,1]: [=][=][=]


## 3. Kiến thức nền

### 3.1 Liệt kê mọi hàng (hợp thành có thứ tự) bằng quay lui

In [2]:
def all_rows(width, bricks):
    out = []
    cur = []
    def go(filled):
        if filled == width:
            out.append(tuple(cur)); return
        for b in bricks:
            if filled + b <= width:
                cur.append(b); go(filled + b); cur.pop()
    go(0)
    return out

print("width=3, bricks=[1,2]:", all_rows(3, [1, 2]))
print("số hàng khi width=10, bricks=1..10:", len(all_rows(10, list(range(1, 11)))), "(= 2^9 hợp thành của 10)")

width=3, bricks=[1,2]: [(1, 1, 1), (1, 2), (2, 1)]
số hàng khi width=10, bricks=1..10: 512 (= 2^9 hợp thành của 10)


### 3.2 Mã hóa hàng bằng bitmask mạch nối

Mạch nối ở vị trí `x` (`1 ≤ x ≤ width − 1`, khoảng cách từ mép trái) ↔ bit `x − 1`. Hai đầu tường (`0` và `width`) **không** tính.

In [3]:
def row_mask(row):
    m, pos = 0, 0
    for b in row[:-1]:                 # mạch sau mỗi viên trừ viên cuối
        pos += b
        m |= 1 << (pos - 1)
    return m

for r in all_rows(3, [1, 2]):
    print(f"  {r}: mạch ở {[i + 1 for i in range(2) if row_mask(r) >> i & 1]}, mask = {row_mask(r):02b}")

  (1, 1, 1): mạch ở [1, 2], mask = 11
  (1, 2): mạch ở [1], mask = 01
  (2, 1): mạch ở [2], mask = 10


### 3.3 Hai hàng hợp nhau ⇔ `a & b == 0`

In [4]:
rows3 = all_rows(3, [1, 2])
for a in rows3:
    print("  ", a, "hợp với:", [b for b in rows3 if row_mask(a) & row_mask(b) == 0])

   (1, 1, 1) hợp với: []
   (1, 2) hợp với: [(2, 1)]
   (2, 1) hợp với: [(1, 2)]


### 3.4 Tổng trên tập con (SOS DP)

Cho `f` trên mọi mask `k` bit, tính `g[m] = Σ_{p ⊆ m} f[p]` cho **mọi** `m` trong `O(2^k · k)`: với từng bit `i`, với mọi `m` có bit `i`, `g[m] += g[m ^ (1 << i)]`.

In [5]:
def subset_sum_transform(f, k):
    g = list(f)
    for i in range(k):
        for m in range(1 << k):
            if m >> i & 1:
                g[m] += g[m ^ (1 << i)]
    return g

f = [1, 2, 3, 4]                       # k = 2
print("g =", subset_sum_transform(f, 2), "| kiểm g[3] = f[0]+f[1]+f[2]+f[3] =", sum(f))

g = [1, 3, 4, 10] | kiểm g[3] = f[0]+f[1]+f[2]+f[3] = 10


## 4. Ý tưởng nền tảng / trực giác

Một bức tường = chọn một hàng cho mỗi tầng, sao cho **hai tầng kề** hợp nhau. Đó là đếm **đường đi dài `height − 1`** trên đồ thị "hàng → hàng hợp nhau".

Tính tay ví dụ 1: các hàng `A = [1,1,1]` (mask `11`), `B = [1,2]` (mask `01`), `C = [2,1]` (mask `10`). Cạnh hợp nhau: `B — C` (vì `01 & 10 = 0`); `A` không hợp với ai (kể cả chính nó, và `B` hợp với `B`? `01 & 01 ≠ 0` — không). Tường cao 2: `B` trên `C` hoặc `C` trên `B` ⇒ 2 ✓.

Ví dụ 2: không có hàng nào (gạch 5 > width 1) ⇒ 0.

**Trường hợp đặc biệt:** `width = 1` với gạch `1`: hàng duy nhất `[1]` có mask `0` (không có mạch) — hợp với chính nó ⇒ đáp án 1 với mọi `height`.

## 5. Cách 1 — Brute Force / baseline: vét cạn mọi bức tường

In [6]:
def wall_brute(height, width, bricks):
    masks = [row_mask(r) for r in all_rows(width, bricks)]
    count = 0
    for wall in product(range(len(masks)), repeat=height):
        if all(masks[wall[i]] & masks[wall[i + 1]] == 0 for i in range(height - 1)):
            count += 1
    return count % MOD

for args, exp in examples:
    print(wall_brute(*args), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(R^height · height)` với `R` = số hàng — bùng nổ. **Nút thắt:** chỉ cần biết hàng **trên cùng** để thêm tầng mới ⇒ DP theo tầng.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái:** `dp[m]` = số tường (đã xây tới tầng hiện tại) có hàng trên cùng mang mask `m` (gộp các hàng có cùng mask — trên thực tế mỗi hàng có mask riêng vì hợp thành ↔ tập vị trí mạch là song ánh).

**Chuyển:** `new[m] = Σ_{p: p & m = 0} dp[p]` — nhân với số hàng có mask `m` (luôn là 0 hoặc 1).

**Khởi tạo:** tầng 1: `dp[m] = 1` với mỗi hàng. **Đáp án:** `Σ dp[m]` sau `height` tầng.

**Tăng tốc:** `p & m = 0` ⇔ `p ⊆ ~m` (trong `k = width − 1` bit) ⇒ `new[m] = S[~m]` với `S` = tổng trên tập con của `dp`. Một lần SOS cho mọi `m`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP theo tầng, duyệt mọi cặp hàng

In [7]:
def wall_dp_pairs(height, width, bricks):
    masks = [row_mask(r) for r in all_rows(width, bricks)]
    if not masks:
        return 0
    dp = [1] * len(masks)
    for _ in range(height - 1):
        dp = [sum(dp[j] for j in range(len(masks)) if masks[i] & masks[j] == 0) % MOD for i in range(len(masks))]
    return sum(dp) % MOD

for args, exp in examples:
    print(wall_dp_pairs(*args), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(height · R²)` — với `R = 512` là `2.6·10⁷` phép, vài giây trong Python.

### 7.2 Cách 3 — Danh sách kề tính sẵn

In [8]:
def wall_dp_adj(height, width, bricks):
    masks = [row_mask(r) for r in all_rows(width, bricks)]
    R = len(masks)
    if R == 0:
        return 0
    adj = [[j for j in range(R) if masks[i] & masks[j] == 0] for i in range(R)]
    dp = [1] * R
    for _ in range(height - 1):
        dp = [sum(dp[j] for j in adj[i]) % MOD for i in range(R)]
    return sum(dp) % MOD

for args, exp in examples:
    print(wall_dp_adj(*args), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(R² + height · E)` với `E` = số cặp hợp nhau (thường ít hơn nhiều so với `R²` khi có nhiều mạch nối).

### 7.3 Cách 4 — DP + tổng trên tập con (SOS)

In [9]:
def wall_dp_sos(height, width, bricks):
    k = width - 1
    full = (1 << k) - 1
    exists = [0] * (1 << k)
    for r in all_rows(width, bricks):
        exists[row_mask(r)] = 1
    if not any(exists):
        return 0
    dp = exists[:]
    for _ in range(height - 1):
        S = dp[:]                                   # S[m] = Σ_{p ⊆ m} dp[p]
        for i in range(k):
            bit = 1 << i
            for m in range(1 << k):
                if m & bit:
                    S[m] += S[m ^ bit]
        dp = [S[full ^ m] % MOD if exists[m] else 0 for m in range(1 << k)]
    return sum(dp) % MOD

for args, exp in examples:
    print(wall_dp_sos(*args), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(height · 2^k · k)` với `k = width − 1 ≤ 9` ⇒ `100 · 512 · 9 ≈ 4.6·10⁵` — nhanh nhất. Khi `width = 1` thì `k = 0`, mảng có 1 phần tử (mask 0), vòng SOS rỗng — vẫn đúng.

### 7.4 Cách 5 — Lũy thừa ma trận kề

Số tường = tổng các phần tử của `M^{height−1} · 𝟙` với `M[i][j] = 1` nếu hàng `i`, `j` hợp nhau. Lũy thừa nhanh: `O(R³ log height)` — chỉ đáng khi `height` rất lớn (ví dụ `10¹⁸`) và `R` nhỏ. Với `R = 512` thì `R³ ≈ 1.3·10⁸` mỗi phép nhân — quá chậm trong Python thuần, nên ta chỉ dùng khi `R` nhỏ.

In [10]:
def mat_mul(A, B):
    n, m, p = len(A), len(B), len(B[0])
    C = [[0] * p for _ in range(n)]
    for i in range(n):
        Ai = A[i]; Ci = C[i]
        for t in range(m):
            a = Ai[t]
            if a:
                Bt = B[t]
                for j in range(p):
                    Ci[j] = (Ci[j] + a * Bt[j]) % MOD
    return C

def wall_matrix_power(height, width, bricks):
    masks = [row_mask(r) for r in all_rows(width, bricks)]
    R = len(masks)
    if R == 0:
        return 0
    M = [[1 if masks[i] & masks[j] == 0 else 0 for j in range(R)] for i in range(R)]
    vec = [[1] for _ in range(R)]
    e = height - 1
    while e:
        if e & 1:
            vec = mat_mul(M, vec)
        M = mat_mul(M, M)
        e >>= 1
    return sum(v[0] for v in vec) % MOD

for args, exp in examples:
    print(wall_matrix_power(*args), "(kỳ vọng", exp, ")")
print("height = 10^18, width = 4, bricks = [1, 2]:", wall_matrix_power(10**18, 4, [1, 2]))

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
height = 10^18, width = 4, bricks = [1, 2]: 2


**Lưu ý thứ tự nhân:** `vec ← M · vec` khi bit đang xét là 1, rồi bình phương `M` — vì `M` giao hoán với chính lũy thừa của nó, thứ tự áp dụng các lũy thừa `M^{2^i}` không quan trọng.

## 8. TRACE từng bước

### 8.1 Trace DP (cách 3) với `height = 4, width = 4, bricks = [1, 2, 3]` (ví dụ tự dựng — đã đối chiếu vét cạn, đối chiếu lại đề gốc)

In [11]:
def trace_dp(height, width, bricks):
    rows = all_rows(width, bricks); masks = [row_mask(r) for r in rows]; R = len(masks)
    adj = [[j for j in range(R) if masks[i] & masks[j] == 0] for i in range(R)]
    for i, r in enumerate(rows):
        print(f"  hàng {i}: {r!s:<12} mask {masks[i]:0{max(1, width - 1)}b} hợp với {adj[i]}")
    dp = [1] * R
    print(f"  tầng 1: dp = {dp}, tổng {sum(dp)}")
    for h in range(2, height + 1):
        dp = [sum(dp[j] for j in adj[i]) % MOD for i in range(R)]
        print(f"  tầng {h}: dp = {dp}, tổng {sum(dp)}")
    return sum(dp) % MOD

r = trace_dp(4, 4, [1, 2, 3])
print("  => đáp án", r, "| vét cạn:", wall_brute(4, 4, [1, 2, 3]))

  hàng 0: (1, 1, 1, 1) mask 111 hợp với []
  hàng 1: (1, 1, 2)    mask 011 hợp với [6]
  hàng 2: (1, 2, 1)    mask 101 hợp với [5]
  hàng 3: (1, 3)       mask 001 hợp với [4, 5, 6]
  hàng 4: (2, 1, 1)    mask 110 hợp với [3]
  hàng 5: (2, 2)       mask 010 hợp với [2, 3, 6]
  hàng 6: (3, 1)       mask 100 hợp với [1, 3, 5]
  tầng 1: dp = [1, 1, 1, 1, 1, 1, 1], tổng 7
  tầng 2: dp = [0, 1, 1, 3, 1, 3, 3], tổng 12
  tầng 3: dp = [0, 3, 3, 7, 3, 7, 7], tổng 30
  tầng 4: dp = [0, 7, 7, 17, 7, 17, 17], tổng 72
  => đáp án 72 | vét cạn: 72


### 8.2 Trace một bước SOS cho ví dụ 1 (`k = 2`)

In [12]:
def trace_sos_step(width, bricks):
    k = width - 1; full = (1 << k) - 1
    exists = [0] * (1 << k)
    for r in all_rows(width, bricks):
        exists[row_mask(r)] = 1
    dp = exists[:]
    print(f"  dp (tầng 1) theo mask: {dict((format(m, f'0{k}b'), dp[m]) for m in range(1 << k))}")
    S = dp[:]
    for i in range(k):
        for m in range(1 << k):
            if m >> i & 1:
                S[m] += S[m ^ (1 << i)]
        print(f"  sau bit {i}: S = {dict((format(m, f'0{k}b'), S[m]) for m in range(1 << k))}")
    new = [S[full ^ m] if exists[m] else 0 for m in range(1 << k)]
    print(f"  tầng 2: new[m] = S[~m] -> {dict((format(m, f'0{k}b'), new[m]) for m in range(1 << k))}, tổng {sum(new)}")

trace_sos_step(3, [1, 2])

  dp (tầng 1) theo mask: {'00': 0, '01': 1, '10': 1, '11': 1}
  sau bit 0: S = {'00': 0, '01': 1, '10': 1, '11': 2}
  sau bit 1: S = {'00': 0, '01': 1, '10': 1, '11': 3}
  tầng 2: new[m] = S[~m] -> {'00': 0, '01': 1, '10': 1, '11': 0}, tổng 2


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def buildWall(self, height: int, width: int, bricks: List[int]) -> int:
        MOD = 10**9 + 7
        k = width - 1
        full = (1 << k) - 1
        exists = [0] * (1 << k)

        stack = [(0, 0)]                            # (độ dài đã xếp, mask mạch nối) — DFS lặp
        while stack:
            filled, mask = stack.pop()
            for b in bricks:
                nf = filled + b
                if nf == width:
                    exists[mask] = 1
                elif nf < width:
                    stack.append((nf, mask | 1 << (nf - 1)))

        dp = exists[:]
        for _ in range(height - 1):
            S = dp[:]                               # S[m] = Σ dp[p] với p ⊆ m
            for i in range(k):
                bit = 1 << i
                for m in range(1 << k):
                    if m & bit:
                        S[m] += S[m ^ bit]
            dp = [S[full ^ m] % MOD if exists[m] else 0 for m in range(1 << k)]
        return sum(dp) % MOD

for args, exp in examples:
    assert Solution().buildWall(*args) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Song ánh hàng ↔ mask.** Một hàng (dãy độ rộng có thứ tự, tổng `width`) xác định duy nhất tập vị trí mạch nối (các tổng tiền tố trừ tổng cuối), và ngược lại tập mạch nối xác định duy nhất dãy độ rộng (hiệu các vị trí liên tiếp). Nên mỗi mask ứng với **tối đa một** hàng — `exists[m] ∈ {0, 1}` (một mask chỉ hợp lệ nếu mọi khoảng cách đều thuộc `bricks`).
2. **Điều kiện vững** giữa hai hàng kề ⇔ không có vị trí mạch chung ⇔ `a & b = 0`.
3. **DP:** quy nạp theo số tầng; `dp[m]` sau tầng `h` đếm đúng số tường cao `h` có hàng trên cùng `m`: mọi tường như vậy tách duy nhất thành (tường cao `h−1` với hàng trên `p`) + hàng `m`, với `p & m = 0`.
4. **SOS:** `p & m = 0` ⇔ `p ⊆ full ^ m`; phép biến đổi tổng tập con tính đúng `Σ_{p ⊆ q} dp[p]` cho mọi `q` (mỗi bit được "mở" một lần — chứng minh chuẩn của zeta transform).
5. **Modulo** áp dụng ở mỗi tầng giữ đúng kết quả mod `10⁹ + 7` (phép cộng tương thích với đồng dư).

## 11. Phân tích Time / Space Complexity

Gọi `k = width − 1 ≤ 9`, `R` = số hàng hợp lệ (`≤ 2^k`), `E` = số cặp hàng hợp nhau, `h = height`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(R^h · h)` | `O(h)` |
| 2. Mọi cặp | `O(h · R²)` | `O(R)` |
| 3. Danh sách kề | `O(R² + h · E)` | `O(E)` |
| 4. SOS | `O(h · 2^k · k)` | `O(2^k)` |
| 5. Lũy thừa ma trận | `O(R³ log h)` | `O(R²)` |

In [14]:
for args in [(100, 10, list(range(1, 11))), (100, 10, [1, 2]), (100, 10, [3, 7, 2]), (100, 9, [1, 4, 5, 6])]:
    row = []
    for name, fn in [("pairs", wall_dp_pairs), ("adj", wall_dp_adj), ("sos", wall_dp_sos), ("Solution", Solution().buildWall)]:
        if name == "pairs" and len(all_rows(args[1], args[2])) > 200:
            row.append("pairs=(bỏ qua)"); continue
        t0 = time.perf_counter(); r = fn(*args); row.append(f"{name}={(time.perf_counter() - t0) * 1000:.0f}ms")
    print(f"h={args[0]}, w={args[1]}, bricks={args[2]} (R={len(all_rows(args[1], args[2]))}) -> {r}: " + ", ".join(row))

h=100, w=10, bricks=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] (R=512) -> 946849490: pairs=(bỏ qua), adj=55ms, sos=20ms, Solution=20ms


h=100, w=10, bricks=[1, 2] (R=89) -> 2: pairs=24ms, adj=1ms, sos=15ms, Solution=15ms


h=100, w=10, bricks=[3, 7, 2] (R=9) -> 240327839: pairs=0ms, adj=0ms, sos=15ms, Solution=20ms


h=100, w=9, bricks=[1, 4, 5, 6] (R=21) -> 503340220: pairs=2ms, adj=1ms, sos=7ms, Solution=7ms


## 12. Bộ test

In [15]:
ALL = {
    "pairs": wall_dp_pairs,
    "adj": wall_dp_adj,
    "sos": wall_dp_sos,
    "matrix": wall_matrix_power,
    "Solution": lambda h, w, b: Solution().buildWall(h, w, b),
}

def check(height, width, bricks, exp=None, brute=True, skip=()):
    ref = wall_brute(height, width, bricks) if brute else wall_dp_sos(height, width, bricks)
    for name, fn in ALL.items():
        if name in skip:
            continue
        got = fn(height, width, list(bricks))
        assert got == ref, (name, height, width, bricks, got, ref)
    if exp is not None:
        assert ref == exp, (height, width, bricks, ref, exp)
    return ref

for args, exp in examples:
    check(*args, exp)
print("Ví dụ đề: OK")

check(1, 1, [1], 1); check(5, 1, [1], 1)       # width 1: hàng [1], không mạch, luôn hợp
check(3, 2, [1], 0)                            # hàng [1,1] có mạch ở 1, tự xung đột
check(1, 2, [1], 1)                            # chiều cao 1: mọi hàng đều được
check(3, 4, [2], 0)                            # chỉ có [2,2], mạch ở 2
check(4, 4, [1, 2, 3])
check(1, 10, list(range(1, 11)), 512, brute=False)
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(300):
    w = rnd.randint(1, 6); b = rnd.sample(range(1, 7), rnd.randint(1, 4)); h = rnd.randint(1, 4)
    if len(all_rows(w, b)) ** h <= 20000:
        check(h, w, b)
print("Ngẫu nhiên nhỏ so với vét cạn: OK")

for _ in range(15):
    w = rnd.randint(1, 10); b = rnd.sample(range(1, 11), rnd.randint(1, 10)); h = rnd.randint(1, 100)
    R_ = len(all_rows(w, b))
    check(h, w, b, brute=False, skip=(("pairs",) if R_ > 120 else ()) + (("matrix",) if R_ > 40 else ()))
print("15 bộ lớn (đối chiếu chéo SOS / adj / Solution): OK")

Ví dụ đề: OK
Edge case: OK
Ngẫu nhiên nhỏ so với vét cạn: OK


15 bộ lớn (đối chiếu chéo SOS / adj / Solution): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tính cả hai đầu tường là mạch nối

In [16]:
def bug_include_ends(height, width, bricks):
    def mask_with_ends(row):
        m, pos = 1, 0                          # SAI: bit 0 cho mép trái
        for b in row:
            pos += b; m |= 1 << pos            # SAI: có cả mạch ở mép phải
        return m
    masks = [mask_with_ends(r) for r in all_rows(width, bricks)]
    dp = [1] * len(masks)
    for _ in range(height - 1):
        dp = [sum(dp[j] for j in range(len(masks)) if masks[i] & masks[j] == 0) for i in range(len(masks))]
    return sum(dp) % MOD

print("ví dụ 1 -> sai:", bug_include_ends(2, 3, [1, 2]), "| đúng:", Solution().buildWall(2, 3, [1, 2]))

ví dụ 1 -> sai: 0 | đúng: 2


**Vì sao sai:** mọi hàng đều có "mạch" ở hai đầu ⇒ không cặp nào hợp nhau. Đề nói rõ "except at the ends of the wall". **Sửa:** chỉ các vị trí `1 … width − 1`.

### Lỗi 2 — Liệt kê hàng như **tổ hợp không thứ tự** (bỏ hoán vị)

In [17]:
def bug_unordered_rows(height, width, bricks):
    rows = set(tuple(sorted(r)) for r in all_rows(width, bricks))     # SAI: [1,2] và [2,1] gộp làm một
    masks = [row_mask(r) for r in rows]
    dp = [1] * len(masks)
    for _ in range(height - 1):
        dp = [sum(dp[j] for j in range(len(masks)) if masks[i] & masks[j] == 0) for i in range(len(masks))]
    return sum(dp) % MOD

print("ví dụ 1 -> sai:", bug_unordered_rows(2, 3, [1, 2]), "| đúng:", Solution().buildWall(2, 3, [1, 2]))

ví dụ 1 -> sai: 0 | đúng: 2


**Vì sao sai:** thứ tự viên gạch trong hàng **quyết định** vị trí mạch; `[1,2]` (mạch ở 1) khác `[2,1]` (mạch ở 2). **Sửa:** quay lui theo thứ tự (hợp thành).

### Lỗi 3 — Cho phép hàng **dài hơn** `width` (cắt bớt) hoặc ngắn hơn

In [18]:
def bug_not_exact(height, width, bricks):
    rows = []
    def go(filled, cur):
        if filled >= width:                    # SAI: chấp nhận vượt width
            rows.append(tuple(cur)); return
        for b in bricks:
            go(filled + b, cur + [b])
    go(0, [])
    def mask(row):
        m, pos = 0, 0
        for b in row[:-1]:
            pos += b
            if pos < width: m |= 1 << (pos - 1)
        return m
    masks = [mask(r) for r in rows]
    dp = [1] * len(masks)
    for _ in range(height - 1):
        dp = [sum(dp[j] for j in range(len(masks)) if masks[i] & masks[j] == 0) for i in range(len(masks))]
    return sum(dp) % MOD

print("(1, 1, [5]) -> sai:", bug_not_exact(1, 1, [5]), "| đúng:", Solution().buildWall(1, 1, [5]))
print("(2, 3, [1, 2]) -> sai:", bug_not_exact(2, 3, [1, 2]), "| đúng:", Solution().buildWall(2, 3, [1, 2]))

(1, 1, [5]) -> sai: 1 | đúng: 0
(2, 3, [1, 2]) -> sai: 4 | đúng: 2


**Sửa:** chỉ nhận hàng có tổng **đúng bằng** `width`.

### Lỗi 4 — Khởi tạo tầng 1 sai: `dp` toàn 1 cho **mọi** mask (kể cả mask không ứng với hàng nào)

In [19]:
def bug_init_all_masks(height, width, bricks):
    k = width - 1; full = (1 << k) - 1
    exists = [0] * (1 << k)
    for r in all_rows(width, bricks):
        exists[row_mask(r)] = 1
    dp = [1] * (1 << k)                        # SAI: phải là exists
    for _ in range(height - 1):
        S = dp[:]
        for i in range(k):
            for m in range(1 << k):
                if m >> i & 1: S[m] += S[m ^ (1 << i)]
        dp = [S[full ^ m] % MOD if exists[m] else 0 for m in range(1 << k)]
    return sum(dp) % MOD

print("(2, 3, [1, 2]) -> sai:", bug_init_all_masks(2, 3, [1, 2]), "| đúng:", Solution().buildWall(2, 3, [1, 2]))
print("(1, 3, [2]) -> sai:", bug_init_all_masks(1, 3, [2]), "| đúng:", Solution().buildWall(1, 3, [2]))

(2, 3, [1, 2]) -> sai: 5 | đúng: 2
(1, 3, [2]) -> sai: 4 | đúng: 0


**Vì sao sai:** mask `00` (hàng không có mạch, tức một viên dài 3) không tồn tại với `bricks = [1,2]`, nhưng vẫn được đếm ở tầng 1 và "nuôi" các tầng sau. **Sửa:** `dp = exists[:]`.

### Lỗi 5 — Gộp tầng bằng phép **nhân** số hàng hợp nhau (coi các tầng độc lập)

In [20]:
def bug_independent(height, width, bricks):
    masks = [row_mask(r) for r in all_rows(width, bricks)]
    pairs = sum(1 for a in masks for b in masks if a & b == 0)
    return len(masks) * (pairs ** (height - 1)) % MOD if masks else 0   # SAI

print("(3, 3, [1, 2]) -> sai:", bug_independent(3, 3, [1, 2]), "| đúng:", Solution().buildWall(3, 3, [1, 2]))

(3, 3, [1, 2]) -> sai: 12 | đúng: 2


**Vì sao sai:** lựa chọn hàng ở tầng `h` **phụ thuộc** hàng tầng `h−1`; không thể nhân số cặp. Tường cao 3 với `[1,2]`: chỉ `B C B` và `C B C` ⇒ 2. **Sửa:** DP theo tầng (đếm đường đi trên đồ thị).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xếp từng hàng/cột, hai hàng kề có ràng buộc, chiều kia nhỏ (≤ 10–12)" ⇒ **DP trên bitmask hàng** (broken profile / row DP).
- Ràng buộc "không trùng vị trí" ⇒ `a & b == 0` ⇒ có thể tăng tốc bằng **SOS** (tổng trên tập con của phần bù).
- Chiều cao rất lớn ⇒ **lũy thừa ma trận** trên đồ thị hàng.
- Liệt kê hàng = **hợp thành có thứ tự**, không phải tổ hợp.
- Mạch ở hai đầu tường không tính — đọc kỹ đề.

## 15. Biến thể của bài

1. **LeetCode 554 — Brick Wall:** cho sẵn tường, tìm đường thẳng đứng cắt ít gạch nhất (đếm vị trí mạch phổ biến nhất).
2. **Đếm cách lát nền `m × n` bằng domino** (LeetCode 790 / bài kinh điển) — DP theo profile bitmask.
3. **Không giới hạn chiều cao** nhưng hỏi chiều cao `h ≤ 10¹⁸` — lũy thừa ma trận (cách 5).
4. **Gạch có số lượng hạn chế** — thêm chiều trạng thái số gạch đã dùng (bùng nổ nhanh).
5. **Chỉ đếm số cách xếp một hàng** = số hợp thành của `width` với phần tử thuộc `bricks` — DP 1 chiều kiểu "climbing stairs".

In [21]:
def count_rows(width, bricks):
    """Biến thể 5: số cách xếp một hàng = số hợp thành của width với phần tử thuộc bricks."""
    f = [1] + [0] * width
    for x in range(1, width + 1):
        f[x] = sum(f[x - b] for b in bricks if b <= x)
    return f[width]

def most_common_joint(wall):
    """Biến thể 1 (LeetCode 554): số gạch ít nhất bị cắt bởi một đường thẳng đứng."""
    from collections import Counter
    c = Counter()
    for row in wall:
        pos = 0
        for b in row[:-1]:
            pos += b; c[pos] += 1
    return len(wall) - (max(c.values()) if c else 0)

rnd_v = random.Random(15)
for _ in range(300):
    w = rnd_v.randint(1, 10); b = rnd_v.sample(range(1, 11), rnd_v.randint(1, 5))
    assert count_rows(w, b) == len(all_rows(w, b))
print("Biến thể 5: width=10, bricks=1..10 ->", count_rows(10, list(range(1, 11))), "hàng")
print("Biến thể 1 (554):", most_common_joint([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]), "(kỳ vọng 2)")

Biến thể 5: width=10, bricks=1..10 -> 512 hàng
Biến thể 1 (554): 2 (kỳ vọng 2)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi tường | `O(R^h h)` | `O(h)` | kiểm thử |
| 2. Mọi cặp | DP tầng | `O(h R²)` | `O(R)` | `R` nhỏ |
| 3. Danh sách kề | DP tầng | `O(R² + h E)` | `O(E)` | dễ hiểu, đủ nhanh |
| 4. SOS | tổng tập con của phần bù | `O(h 2^k k)` | `O(2^k)` | **nộp bài** |
| 5. Lũy thừa ma trận | `M^{h−1}` | `O(R³ log h)` | `O(R²)` | `h` khổng lồ, `R` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().buildWall(3, 3, [1, 2])`?

**Bài 2 (tính tay):** `width = 4, bricks = [1, 3]`: liệt kê các hàng, mask, đồ thị hợp nhau và số tường cao 2.

**Bài 3 (điền TODO):** Hoàn thiện `walls_with_top(height, width, bricks)` — trả về `dict` {hàng trên cùng (tuple) → số tường}.

**Bài 4 (tự viết lại):** Viết lại cách 3 dùng **ma trận kề + nhân vector** (một bước `dp ← M · dp` mỗi tầng), kiểm với SOS.

**Bài 5 (mini challenge):** Đếm số tường vững **có tính đối xứng gương** (mỗi hàng đọc từ phải sang trái giống từ trái sang phải, tức hàng là palindrome). Kiểm bằng vét cạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def walls_with_top(height, width, bricks):
    # TODO: DP như cách 3, trả về {rows[i]: dp[i]}
    pass

# Bài 4 — TODO
def wall_matvec(height, width, bricks):
    # TODO: M[i][j] = 1 nếu hợp nhau; dp = M · dp, (height - 1) lần
    pass

# Bài 5 — TODO
def wall_palindromic_rows(height, width, bricks):
    # TODO: chỉ giữ hàng r với r == r[::-1], rồi DP như thường
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Chỉ `B = [1,2]` và `C = [2,1]` có cạnh với nhau.
- Mức 2: Tường cao 3 phải xen kẽ `B, C`.
- Mức 3: `BCB`, `CBC` ⇒ 2.

**Bài 2**
- Mức 1: Hàng `[1,1,1,1]`, `[1,3]`, `[3,1]`.
- Mức 2: Mask `111`, `001`, `100`; `[1,3]` hợp `[3,1]`.
- Mức 3: Tường cao 2: 2 cách.

**Bài 3**
- Mức 1: Giữ `dp` theo chỉ số hàng.
- Mức 2: `dict(zip(rows, dp))`.
- Mức 3: Tổng các giá trị = đáp án.

**Bài 4**
- Mức 1: Ma trận `R × R`.
- Mức 2: `dp_new[i] = Σ_j M[i][j] dp[j]`.
- Mức 3: So với `wall_dp_sos`.

**Bài 5**
- Mức 1: Lọc `rows` theo `r == r[::-1]`.
- Mức 2: Phần DP giữ nguyên.
- Mức 3: Vét cạn: liệt kê mọi tường, kiểm từng hàng palindrome + vững.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = Solution().buildWall(3, 3, [1, 2])
print("Bài 1:", r1); assert r1 == 2 == wall_brute(3, 3, [1, 2])

# Bài 2
rows2 = all_rows(4, [1, 3])
print("Bài 2:", [(r, format(row_mask(r), "03b")) for r in rows2], "| tường cao 2:", Solution().buildWall(2, 4, [1, 3]))
assert Solution().buildWall(2, 4, [1, 3]) == 2

# Bài 3
def walls_with_top_sol(height, width, bricks):
    rows = all_rows(width, bricks); masks = [row_mask(r) for r in rows]; R = len(rows)
    adj = [[j for j in range(R) if masks[i] & masks[j] == 0] for i in range(R)]
    dp = [1] * R
    for _ in range(height - 1):
        dp = [sum(dp[j] for j in adj[i]) % MOD for i in range(R)]
    return dict(zip(rows, dp))

rnd = random.Random(3)
for _ in range(200):
    w = rnd.randint(1, 7); b = rnd.sample(range(1, 8), rnd.randint(1, 4)); h = rnd.randint(1, 20)
    assert sum(walls_with_top_sol(h, w, b).values()) % MOD == Solution().buildWall(h, w, b)
print("Bài 3 OK, ví dụ:", walls_with_top_sol(3, 4, [1, 2, 3]))

Bài 1: 2
Bài 2: [((1, 1, 1, 1), '111'), ((1, 3), '001'), ((3, 1), '100')] | tường cao 2: 2
Bài 3 OK, ví dụ: {(1, 1, 1, 1): 0, (1, 1, 2): 3, (1, 2, 1): 3, (1, 3): 7, (2, 1, 1): 3, (2, 2): 7, (3, 1): 7}


In [24]:
# Bài 4
def wall_matvec_sol(height, width, bricks):
    masks = [row_mask(r) for r in all_rows(width, bricks)]; R = len(masks)
    if R == 0: return 0
    M = [[1 if masks[i] & masks[j] == 0 else 0 for j in range(R)] for i in range(R)]
    dp = [1] * R
    for _ in range(height - 1):
        dp = [sum(M[i][j] * dp[j] for j in range(R)) % MOD for i in range(R)]
    return sum(dp) % MOD

# Bài 5
def wall_palindromic_rows_sol(height, width, bricks):
    rows = [r for r in all_rows(width, bricks) if r == r[::-1]]
    masks = [row_mask(r) for r in rows]; R = len(rows)
    if R == 0: return 0
    dp = [1] * R
    for _ in range(height - 1):
        dp = [sum(dp[j] for j in range(R) if masks[i] & masks[j] == 0) % MOD for i in range(R)]
    return sum(dp) % MOD

def palindromic_brute(height, width, bricks):
    rows = [r for r in all_rows(width, bricks) if r == r[::-1]]; masks = [row_mask(r) for r in rows]
    return sum(1 for wall in product(range(len(rows)), repeat=height)
               if all(masks[wall[i]] & masks[wall[i + 1]] == 0 for i in range(height - 1)))

rnd = random.Random(5)
for _ in range(200):
    w = rnd.randint(1, 8); b = rnd.sample(range(1, 9), rnd.randint(1, 4)); h = rnd.randint(1, 4)
    assert wall_matvec_sol(h, w, b) == wall_dp_sos(h, w, b)
    if len([r for r in all_rows(w, b) if r == r[::-1]]) ** h <= 20000:
        assert wall_palindromic_rows_sol(h, w, b) == palindromic_brute(h, w, b)
print("Bài 4, Bài 5 OK; ví dụ:", wall_matvec_sol(4, 4, [1, 2, 3]), wall_palindromic_rows_sol(3, 6, [1, 2, 4]))

Bài 4, Bài 5 OK; ví dụ: 72 10


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "DP hàng bằng bitmask + SOS":**

```python
k = width - 1; full = (1 << k) - 1
exists = [0] * (1 << k)                  # mask mạch nối của mọi hàng hợp lệ
... DFS liệt kê hàng, exists[mask] = 1
dp = exists[:]
for _ in range(height - 1):
    S = dp[:]
    for i in range(k):
        for m in range(1 << k):
            if m >> i & 1: S[m] += S[m ^ (1 << i)]
    dp = [S[full ^ m] % MOD if exists[m] else 0 for m in range(1 << k)]
return sum(dp) % MOD
```

**Checklist:**
- [ ] Mạch nối chỉ ở vị trí `1 … width − 1` (không tính hai đầu)?
- [ ] Hàng là **hợp thành có thứ tự**, tổng đúng bằng `width`?
- [ ] Khởi tạo `dp = exists`?
- [ ] `width = 1` (k = 0) vẫn chạy đúng?
- [ ] Modulo ở mỗi tầng?
- [ ] Chữ ký đúng: `def buildWall(self, height: int, width: int, bricks: List[int]) -> int`? (đối chiếu đề gốc nếu có Premium)